Pariwise Bivarite correlation of variables and MLR with ARMA(1,1)

In [3]:
"""
Transparent MLR analysis.

Primary models:
  Model 1: CHL ~ Niño 3.4 + DMI + SST
  Model 2: CHL ~ Niño 3.4 + DMI

The full monthly calendar is retained for SARIMAX/state-space regression.
Missing CHL is not imputed. Complete-case OLS/HAC/GLSAR are sensitivity analyses,
not the primary time-series inference.

"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from docx import Document

# ----------------------- EDIT THESE PATHS -----------------------
INPUT_FILE = Path("Standardized_SST_CHL_ENSO_IOD_282months.xlsx")
SHEET_NAME = "Standardized_Data"
OUTPUT_DIR = Path("output")
FIGURE_DIR = Path("output")
# Limited, transparent candidate set
P_CANDIDATES = range(0, 4)       # AR order 0,1,2,3
Q_CANDIDATES = range(0, 3)       # MA order 0,1,2
CRITERION = "AIC"                # "AIC" or "BIC"
HAC_MAXLAGS = 12                 # sensitivity only
BOOTSTRAP_BLOCK = 12             # calendar months
N_BOOTSTRAP = 3000
SEED = 20261009
DIAGNOSTIC_LAGS = 24
# ----------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUT_XLSX = OUTPUT_DIR / "MLR_results_revised.xlsx"
OUT_DOCX = OUTPUT_DIR / "MLR_tables_revised.docx"
OUT_PDF = FIGURE_DIR / "MLR_residual_diagnostics_revised.pdf"

MODELS = {
    "Model 1 (with SST)": ["Nino", "DMI", "SST"],
    "Model 2 (without SST)": ["Nino", "DMI"],
}
LABEL = {"Nino": "Niño 3.4", "DMI": "DMI", "SST": "SST", "CHL": "CHL"}
FINAL_ESTIMATOR = "State-space regression with ARMA errors"


def pstr(p):
    if pd.isna(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_FILE}\nEdit INPUT_FILE at the top of the script."
        )
    d = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
    d.columns = d.columns.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    if "Month-Year" not in d:
        raise ValueError(f"Missing 'Month-Year'. Columns found: {d.columns.tolist()}")
    d["Date"] = pd.to_datetime(d["Month-Year"], format="%b-%Y", errors="coerce")
    if d["Date"].isna().any():
        raise ValueError("Some Month-Year values could not be parsed as %b-%Y.")
    if d["Date"].duplicated().any():
        raise ValueError("Duplicate months detected. Resolve duplicates before analysis.")
    d = d.sort_values("Date").rename(columns={
        "Niño 3.4": "Nino", "Nino 3.4": "Nino",
        "SST (°C)": "SST", "CHL (mg/m^3)": "CHL", "CHL (mg/m³)": "CHL"
    }).set_index("Date")
    needed = ["CHL", "Nino", "DMI", "SST"]
    missing = [c for c in needed if c not in d.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}; found {d.columns.tolist()}")
    d = d[needed].apply(pd.to_numeric, errors="coerce").asfreq("MS")
    bad = d["CHL"].notna() & d[["Nino", "DMI", "SST"]].isna().any(axis=1)
    if bad.any():
        raise ValueError("Predictor missing where CHL is observed: " +
                         ", ".join(d.index[bad].strftime("%Y-%m")))
    complete = d.dropna(subset=needed).copy()
    return d, complete


def fit_sarimax(cal, predictors, order):
    """Fit on the full calendar; no missing CHL values are filled/imputed."""
    y = cal["CHL"].astype(float)
    x = cal[predictors].astype(float).copy() if predictors else None
    if x is not None:
        # SARIMAX requires finite exog. Set exog to 0 only where y is missing.
        # Such rows have no observed response likelihood contribution.
        x.loc[y.isna(), :] = 0.0
        if x.isna().any().any():
            raise ValueError("Exogenous predictors contain missing values.")
    mod = SARIMAX(
        y, exog=x, order=(order[0], 0, order[1]), trend="c",
        enforce_stationarity=True, enforce_invertibility=True, missing="none"
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = mod.fit(disp=False, maxiter=1000)
    converged = bool(res.mle_retvals.get("converged", False))
    warn = " | ".join(str(w.message) for w in caught)
    return res, converged, warn


def innovations_on_calendar(res, cal):
    values = np.asarray(res.filter_results.standardized_forecasts_error[0], dtype=float)
    if len(values) != len(cal):
        raise RuntimeError("Innovation vector length differs from calendar length.")
    s = pd.Series(values, index=cal.index, name="standardized_innovation")
    s.loc[cal["CHL"].isna()] = np.nan
    return s.replace([np.inf, -np.inf], np.nan)


def calendar_acf(s, nlags):
    """ACF with calendar positions retained; descriptive with missing data."""
    values = acf(s.to_numpy(dtype=float), nlags=nlags, fft=False, missing="conservative")
    valid = np.isfinite(s.to_numpy(dtype=float))
    pairs = [int(valid.sum())]
    for lag in range(1, nlags + 1):
        pairs.append(int(np.sum(valid[:-lag] & valid[lag:])))
    return values, np.asarray(pairs)


def coefficients_table(res, model, estimator, n_calendar, n_observed):
    ci = res.conf_int()
    rows = []
    for term in res.params.index:
        if term == "sigma2":
            continue
        name = "Intercept" if term in ("intercept", "const") else LABEL.get(term, term)
        rows.append({
            "Model": model, "Estimator": estimator, "Term": name,
            "Coefficient": float(res.params[term]), "SE": float(res.bse[term]),
            "z/t": float(res.params[term] / res.bse[term]),
            "p-value": float(res.pvalues[term]),
            "CI95_low": float(ci.loc[term].iloc[0]),
            "CI95_high": float(ci.loc[term].iloc[1]),
            "Calendar_months": n_calendar, "Observed_CHL": n_observed
        })
    return pd.DataFrame(rows)


def vif_table(comp, predictors, model):
    x = sm.add_constant(comp[predictors], has_constant="add")
    a = x.to_numpy()
    return pd.DataFrame([{
        "Model": model, "Predictor": LABEL[v],
        "VIF": float(variance_inflation_factor(a, i + 1)),
        "N complete cases": len(comp)
    } for i, v in enumerate(predictors)])


def block_boot_corr_ci(x, y, block=12, nboot=3000, seed=1):
    """Moving-block bootstrap on full calendar arrays, preserving missing slots."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) != len(y):
        raise ValueError("Correlation arrays must have same length.")
    n = len(x)
    if n < block:
        return np.nan, np.nan, 0
    rng = np.random.default_rng(seed)
    nblocks = int(np.ceil(n / block))
    offsets = np.arange(block)
    vals = []
    for _ in range(nboot):
        starts = rng.integers(0, n - block + 1, size=nblocks)
        ix = (starts[:, None] + offsets).ravel()[:n]
        xb, yb = x[ix], y[ix]
        ok = np.isfinite(xb) & np.isfinite(yb)
        if ok.sum() >= 10 and np.std(xb[ok]) > 0 and np.std(yb[ok]) > 0:
            vals.append(np.corrcoef(xb[ok], yb[ok])[0, 1])
    if len(vals) < 0.8 * nboot:
        warnings.warn(f"Only {len(vals)} valid bootstrap replicates.")
    if not vals:
        return np.nan, np.nan, 0
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi), len(vals)


def approximate_neff_corr(cal, a, b, maxlag=12):
    """
    Approximate correlation p-value using Neff ~= n/(1+2 sum rho_x(k)rho_y(k)).
    This is not exact for arbitrary missingness/nonstationarity; report as approximate.
    """
    pair = cal[[a, b]].dropna()
    n = len(pair)
    r = float(pair[a].corr(pair[b])) if n >= 3 else np.nan
    if n < 5 or not np.isfinite(r):
        return n, r, np.nan, np.nan
    ax, _ = calendar_acf(cal[a], maxlag)
    ay, _ = calendar_acf(cal[b], maxlag)
    total = sum(ax[k] * ay[k] for k in range(1, maxlag + 1)
                if np.isfinite(ax[k]) and np.isfinite(ay[k]))
    denom = 1 + 2 * total
    neff = float(np.clip(n / denom, 5, n)) if denom > 0 else float(n)
    t = r * np.sqrt((neff - 2) / max(1e-12, 1 - r*r))
    p = float(2 * stats.t.sf(abs(t), max(1, neff - 2)))
    return n, r, neff, p


def diagnostics_figure(cal, fits):
    model = "Model 1 (with SST)"
    ols_resid = fits[model]["ols"].resid.to_numpy(float)
    innov = innovations_on_calendar(fits[model]["arma"], cal)
    r, pairs = calendar_acf(innov, DIAGNOSTIC_LAGS)

    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    plot_acf(ols_resid, lags=DIAGNOSTIC_LAGS, alpha=.05, zero=False, ax=ax[0, 0])
    ax[0, 0].set_title("OLS residual ACF (complete-case row lags)")
    ax[0, 0].set_xlabel("Row lag; not always calendar months")
    plot_pacf(ols_resid, lags=DIAGNOSTIC_LAGS, alpha=.05, zero=False,
              method="ywm", ax=ax[0, 1])
    ax[0, 1].set_title("OLS residual PACF (complete-case row lags)")
    ax[0, 1].set_xlabel("Row lag; not always calendar months")

    lags = np.arange(1, DIAGNOSTIC_LAGS + 1)
    ax[1, 0].stem(lags, r[1:], basefmt=" ")
    band = 1.96 / np.sqrt(max(1, int(np.nanmedian(pairs[1:]))))
    ax[1, 0].axhline(band, ls="--", lw=1)
    ax[1, 0].axhline(-band, ls="--", lw=1)
    ax[1, 0].axhline(0, color="black", lw=.7)
    ax[1, 0].set_ylim(-.5, 1)
    ax[1, 0].set_title("Final-model innovation ACF (calendar positions retained)")
    ax[1, 0].set_xlabel("Calendar lag (months)")
    ax[1, 0].set_ylabel("ACF")
    ax[1, 0].text(.02, .96, "Dashed band is approximate only\nNot a formal test",
                  transform=ax[1, 0].transAxes, va="top", fontsize=9)

    ax[1, 1].plot(cal.index, innov, lw=.8)
    ax[1, 1].axhline(0, color="black", lw=.7)
    ax[1, 1].set_title("Final-model standardized innovations")
    ax[1, 1].set_xlabel("Month")
    ax[1, 1].set_ylabel("Standardized innovation")
    fig.suptitle("Residual diagnostics — inspect alongside numerical model checks")
    fig.tight_layout()
    fig.savefig(OUT_PDF, bbox_inches="tight")
    plt.close(fig)

    return pd.DataFrame([{
        "Calendar lag": k, "Innovation ACF": float(r[k]),
        "Available calendar pairs": int(pairs[k]),
        "Note": "Descriptive ACF with missing positions preserved; no formal p-value"
    } for k in [1, 2, 3, 4, 6, 12, 18, 24]])


def add_table(doc, title, data):
    doc.add_heading(title, level=2)
    view = data.copy()
    table = doc.add_table(rows=1, cols=len(view.columns))
    table.style = "Table Grid"
    for j, c in enumerate(view.columns):
        table.rows[0].cells[j].text = str(c)
    for _, row in view.iterrows():
        cells = table.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = "NA" if pd.isna(v) else (f"{v:.4g}" if isinstance(v, (float, np.floating)) else str(v))


def main():
    cal, comp = load_data()
    print(f"Calendar months={len(cal)}; observed CHL={cal.CHL.notna().sum()}; "
          f"complete cases={len(comp)}; missing CHL={cal.CHL.isna().sum()}")
    print("Missing CHL months:", cal.index[cal.CHL.isna()].strftime("%Y-%m").tolist())

    # Select a limited ARMA order on Model 1; keep convergence/warnings visible.
    search_rows = []
    for p in P_CANDIDATES:
        for q in Q_CANDIDATES:
            try:
                res, conv, warn = fit_sarimax(cal, MODELS["Model 1 (with SST)"], (p, q))
                search_rows.append({"AR": p, "MA": q, "AIC": res.aic, "BIC": res.bic,
                                    "Converged": conv, "Warnings": warn, "LogLik": res.llf})
            except Exception as e:
                search_rows.append({"AR": p, "MA": q, "AIC": np.nan, "BIC": np.nan,
                                    "Converged": False, "Warnings": f"FAILED: {e}", "LogLik": np.nan})
    search = pd.DataFrame(search_rows)
    valid = search[search.Converged & np.isfinite(search[CRITERION])]
    if valid.empty:
        raise RuntimeError("No candidate model converged. Inspect warnings; do not report inference.")
    chosen = valid.sort_values(CRITERION).iloc[0]
    order = (int(chosen.AR), int(chosen.MA))
    print("\nARMA order search:\n", search.sort_values(CRITERION).to_string(index=False))
    print(f"\nProvisional order by {CRITERION}: ARMA{order}")

    coef_list, vif_list, beta_list, stats_list, comparison_list = [], [], [], [], []
    fits = {}
    for model, predictors in MODELS.items():
        X = sm.add_constant(comp[predictors], has_constant="add")
        y = comp.CHL
        ols = sm.OLS(y, X).fit()
        hac = sm.OLS(y, X).fit(cov_type="HAC",
            cov_kwds={"maxlags": min(HAC_MAXLAGS, max(1, len(comp)//5))})
        # Sensitivity only: complete cases compress missing calendar months.
        glsar_model = sm.GLSAR(y, X, rho=1)
        glsar = glsar_model.iterative_fit(maxiter=50)
        arma, converged, fit_warnings = fit_sarimax(cal, predictors, order)
        if not converged:
            warnings.warn(f"{model}: ARMA fit did not converge.")
        if fit_warnings:
            warnings.warn(f"{model} ARMA warnings: {fit_warnings}")
        fits[model] = {"ols": ols, "hac": hac, "glsar": glsar,
                       "arma": arma, "converged": converged}

        for name, result in [
            ("OLS complete-case sensitivity", ols),
            ("OLS+HAC complete-case sensitivity", hac),
            ("GLSAR AR(1) compressed-calendar sensitivity", glsar),
        ]:
            coef_list.append(coefficients_table(result, model, name, len(cal), len(comp)))
        coef_list.append(coefficients_table(arma, model, FINAL_ESTIMATOR, len(cal),
                                            int(cal.CHL.notna().sum())))
        vif_list.append(vif_table(comp, predictors, model))
        for v in predictors:
            beta_list.append({"Model": model, "Predictor": LABEL[v],
                "Standardized beta (descriptive)": float(
                    arma.params[v] * comp[v].std(ddof=1) / comp.CHL.std(ddof=1))})

        # OLS Ljung-Box is only for the compressed complete-case row sequence.
        lb = acorr_ljungbox(ols.resid, lags=[6, 12], model_df=0, return_df=True)
        innov = innovations_on_calendar(arma, cal)
        ar, pairs = calendar_acf(innov, DIAGNOSTIC_LAGS)
        # Error-only model uses same order and same calendar.
        null, null_conv, null_warn = fit_sarimax(cal, [], order)
        lr = max(0.0, float(2 * (arma.llf - null.llf)))
        lr_p = float(stats.chi2.sf(lr, len(predictors)))
        stats_list.append({
            "Model": model, "Predictors": " + ".join(LABEL[v] for v in predictors),
            "Calendar months": len(cal), "Observed CHL": int(cal.CHL.notna().sum()),
            "Complete cases": len(comp), "ARMA order": str(order),
            "ARMA converged": converged, "ARMA warnings": fit_warnings,
            "ARMA AIC": float(arma.aic), "ARMA BIC": float(arma.bic),
            "OLS R2 (OLS only)": float(ols.rsquared),
            "OLS adjusted R2 (OLS only)": float(ols.rsquared_adj),
            "OLS DW (compressed row order)": float(durbin_watson(ols.resid)),
            "OLS Ljung-Box p row lag 6": float(lb.loc[6, "lb_pvalue"]),
            "OLS Ljung-Box p row lag 12": float(lb.loc[12, "lb_pvalue"]),
            "Final innovation ACF calendar lag 12": float(ar[12]),
            "Final innovation ACF calendar lag 24": float(ar[24]),
            "Innovation pairs at calendar lag 12": int(pairs[12]),
            "Innovation pairs at calendar lag 24": int(pairs[24]),
            "LR statistic vs error-only": lr, "LR df": len(predictors),
            "LR p vs error-only": lr_p, "Error-only converged": null_conv,
            "Error-only warnings": null_warn,
        })
        # OLS/GLSAR use "const"; SARIMAX with trend="c" typically uses "intercept".
        arma_intercept_name = next(
            (name for name in ("intercept", "const") if name in arma.params.index),
            None
        )
        if arma_intercept_name is None:
            raise KeyError(
                "Could not find the SARIMAX intercept parameter. "
                f"Available parameters: {list(arma.params.index)}"
            )

        for term in ["const"] + predictors:
            arma_term = arma_intercept_name if term == "const" else term
            comparison_list.append({
                "Model": model, "Term": "Intercept" if term == "const" else LABEL[term],
                "OLS coef": float(ols.params[term]), "OLS p": float(ols.pvalues[term]),
                "HAC p (sensitivity)": float(hac.pvalues[term]),
                "GLSAR coef (sensitivity)": float(glsar.params[term]),
                "GLSAR p (sensitivity)": float(glsar.pvalues[term]),
                "ARMA coef": float(arma.params[arma_term]),
                "ARMA p": float(arma.pvalues[arma_term]),
            })

    coefficients = pd.concat(coef_list, ignore_index=True)
    vifs = pd.concat(vif_list, ignore_index=True)
    betas = pd.DataFrame(beta_list)
    model_stats = pd.DataFrame(stats_list)
    estimator_comparison = pd.DataFrame(comparison_list)

    # One scientifically motivated interaction; interpret cautiously.
    centered = cal.copy()
    for v in ["Nino", "DMI", "SST"]:
        centered[v] = centered[v] - comp[v].mean()
    centered["Nino_x_DMI"] = centered.Nino * centered.DMI
    main, main_conv, main_warn = fit_sarimax(centered, MODELS["Model 1 (with SST)"], order)
    inter, inter_conv, inter_warn = fit_sarimax(
        centered, MODELS["Model 1 (with SST)"] + ["Nino_x_DMI"], order)
    lr_int = max(0.0, float(2 * (inter.llf - main.llf)))
    ci = inter.conf_int().loc["Nino_x_DMI"]
    interaction = pd.DataFrame([{
        "Main effects AIC": main.aic, "Interaction AIC": inter.aic,
        "Main effects BIC": main.bic, "Interaction BIC": inter.bic,
        "Interaction coefficient": inter.params["Nino_x_DMI"],
        "Interaction SE": inter.bse["Nino_x_DMI"],
        "CI95 low": ci.iloc[0], "CI95 high": ci.iloc[1],
        "Wald p": inter.pvalues["Nino_x_DMI"], "LR statistic": lr_int,
        "LR p": stats.chi2.sf(lr_int, 1), "Both converged": main_conv and inter_conv,
        "Main warnings": main_warn, "Interaction warnings": inter_warn,
        "Note": "Do not retain solely on the basis of exploratory significance."
    }])

    corr_rows = []
    pairs_to_test = [("Nino", "CHL"), ("DMI", "CHL"), ("SST", "CHL"),
                     ("Nino", "DMI"), ("Nino", "SST"), ("DMI", "SST")]
    for i, (a, b) in enumerate(pairs_to_test):
        n, r, neff, p = approximate_neff_corr(cal, a, b)
        lo, hi, valid_boot = block_boot_corr_ci(
            cal[a].to_numpy(), cal[b].to_numpy(), BOOTSTRAP_BLOCK,
            N_BOOTSTRAP, SEED + i)
        corr_rows.append({
            "Pair": f"{LABEL[a]} vs {LABEL[b]}", "N complete pairs": n,
            "Pearson r": r, "Approximate Neff": neff,
            "Approximate Neff p (not exact)": p,
            "Block bootstrap CI low": lo, "Block bootstrap CI high": hi,
            "Block length months": BOOTSTRAP_BLOCK, "Valid bootstrap reps": valid_boot,
            "Caution": "Approximate Neff p-value; bootstrap retains calendar gaps."
        })
    correlations = pd.DataFrame(corr_rows)

    innovation_diagnostics = diagnostics_figure(cal, fits)
    audit = cal.copy()
    audit.insert(0, "Date", audit.index)
    audit["CHL observed"] = audit.CHL.notna()

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
        audit.to_excel(w, sheet_name="Data_audit", index=False)
        search.to_excel(w, sheet_name="ARMA_order_search", index=False)
        coefficients.to_excel(w, sheet_name="Coefficients", index=False)
        model_stats.to_excel(w, sheet_name="Model_statistics", index=False)
        vifs.to_excel(w, sheet_name="VIF_by_predictor", index=False)
        betas.to_excel(w, sheet_name="Standardized_betas", index=False)
        estimator_comparison.to_excel(w, sheet_name="Estimator_sensitivity", index=False)
        interaction.to_excel(w, sheet_name="ENSO_IOD_interaction", index=False)
        correlations.to_excel(w, sheet_name="Correlation_robustness", index=False)
        innovation_diagnostics.to_excel(w, sheet_name="Innovation_ACF", index=False)

    doc = Document()
    doc.add_heading("MLR results and time-series diagnostics", level=1)
    doc.add_paragraph(
        "Primary inference uses state-space regression with ARMA errors on the full "
        "monthly calendar; missing CHL observations remain missing. OLS/HAC/GLSAR use "
        "complete cases and are sensitivity analyses. Their lags are row lags across "
        "a compressed dataset, not consistently calendar-month lags. The innovation "
        "ACF preserves missing calendar positions; its approximate bands are descriptive, "
        "not a formal calibrated test. Model adequacy must be judged from all diagnostics."
    )
    for model in MODELS:
        final = coefficients[(coefficients.Model == model) &
                             (coefficients.Estimator == FINAL_ESTIMATOR)]
        add_table(doc, f"Final coefficients: {model}", final)
        add_table(doc, f"Model statistics: {model}", model_stats[model_stats.Model == model])
    add_table(doc, "VIF by predictor", vifs)
    add_table(doc, "ENSO × IOD interaction", interaction)
    add_table(doc, "Correlation robustness", correlations)
    doc.save(OUT_DOCX)

    print("\nMODEL STATISTICS\n", model_stats.to_string(index=False))
    print("\nVIF\n", vifs.to_string(index=False))
    print("\nINTERACTION\n", interaction.to_string(index=False))
    print("\nCORRELATIONS\n", correlations.to_string(index=False))
    print(f"\nSaved:\n{OUT_XLSX}\n{OUT_DOCX}\n{OUT_PDF}")
    print("\nImportant: do not claim ARMA resolved autocorrelation solely because early ACF spikes are small.")


if __name__ == "__main__":
    main()


Calendar months=282; observed CHL=269; complete cases=269; missing CHL=13
Missing CHL months: ['2003-06', '2012-05', '2012-06', '2012-07', '2012-08', '2013-05', '2014-06', '2014-08', '2015-06', '2015-07', '2015-08', '2015-12', '2016-04']

ARMA order search:
  AR  MA        AIC        BIC  Converged Warnings      LogLik
  1   1 635.662782 661.156132       True          -310.831391
  2   0 635.893400 661.386749       True          -310.946700
  3   2 636.214289 672.633360       True          -308.107144
  3   0 637.360739 666.495996       True          -310.680369
  1   2 637.453121 666.588377       True          -310.726560
  2   1 637.527956 666.663213       True          -310.763978
  2   2 638.975164 671.752328       True          -310.487582
  3   1 639.330422 672.107586       True          -310.665211
  1   0 641.125160 662.976603       True          -314.562580
  0   2 661.787584 687.280934       True          -323.893792
  0   1 684.286986 706.138429       True          -336.1434

MLR Model Residual Diagnostic figure Generation

In [3]:
"""
Standalone Figure Generation Script for Model Residual Diagnostics
- Outputs publication-ready PDF & PNG
- Uses Times New Roman font, clear panel labels (a, b, c, d), and larger fonts.
"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ----------------------- GLOBAL FONT & STYLE SETUP -----------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.titlesize": 16
})

# ----------------------- EDIT THESE PATHS -----------------------
INPUT_FILE = Path("Standardized_SST_CHL_ENSO_IOD_282months.xlsx")
SHEET_NAME = "Standardized_Data"
FIGURE_DIR = Path("output")
DIAGNOSTIC_LAGS = 24
# ----------------------------------------------------------------

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUT_PDF = FIGURE_DIR / "MLR_residual_diagnostics_revised_2.pdf"
OUT_PNG = FIGURE_DIR / "MLR_residual_diagnostics_revised_2.png"


def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}\nEdit INPUT_FILE at the top of the script.")
    d = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
    d.columns = d.columns.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    if "Month-Year" not in d:
        raise ValueError(f"Missing 'Month-Year'. Columns found: {d.columns.tolist()}")
    d["Date"] = pd.to_datetime(d["Month-Year"], format="%b-%Y", errors="coerce")
    d = d.sort_values("Date").rename(columns={
        "Niño 3.4": "Nino", "Nino 3.4": "Nino",
        "SST (°C)": "SST", "CHL (mg/m^3)": "CHL", "CHL (mg/m³)": "CHL"
    }).set_index("Date")
    needed = ["CHL", "Nino", "DMI", "SST"]
    d = d[needed].apply(pd.to_numeric, errors="coerce").asfreq("MS")
    complete = d.dropna(subset=needed).copy()
    return d, complete


def fit_sarimax(cal, predictors, order=(1, 1)):
    y = cal["CHL"].astype(float)
    x = cal[predictors].astype(float).copy() if predictors else None
    if x is not None:
        x.loc[y.isna(), :] = 0.0
    mod = SARIMAX(
        y, exog=x, order=(order[0], 0, order[1]), trend="c",
        enforce_stationarity=True, enforce_invertibility=True, missing="none"
    )
    res = mod.fit(disp=False, maxiter=1000)
    return res


def innovations_on_calendar(res, cal):
    values = np.asarray(res.filter_results.standardized_forecasts_error[0], dtype=float)
    s = pd.Series(values, index=cal.index, name="standardized_innovation")
    s.loc[cal["CHL"].isna()] = np.nan
    return s.replace([np.inf, -np.inf], np.nan)


def calendar_acf(s, nlags):
    values = acf(s.to_numpy(dtype=float), nlags=nlags, fft=False, missing="conservative")
    valid = np.isfinite(s.to_numpy(dtype=float))
    pairs = [int(valid.sum())]
    for lag in range(1, nlags + 1):
        pairs.append(int(np.sum(valid[:-lag] & valid[lag:])))
    return values, np.asarray(pairs)


def main():
    cal, comp = load_data()
    
    # Fit Model 1 (with SST) using default ARMA(1,1)
    predictors = ["Nino", "DMI", "SST"]
    X = sm.add_constant(comp[predictors], has_constant="add")
    ols = sm.OLS(comp.CHL, X).fit()
    arma = fit_sarimax(cal, predictors, order=(1, 1))

    ols_resid = ols.resid.to_numpy(float)
    innov = innovations_on_calendar(arma, cal)
    r, pairs = calendar_acf(innov, DIAGNOSTIC_LAGS)

    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    
    # Panel (a): OLS residual ACF
    plot_acf(ols_resid, lags=DIAGNOSTIC_LAGS, alpha=.05, zero=False, ax=ax[0, 0])
    ax[0, 0].set_title("(a) OLS residual ACF (complete-case row lags)", fontweight="bold")
    ax[0, 0].set_xlabel("Row lag (complete cases)")
    ax[0, 0].set_ylabel("Autocorrelation")

    # Panel (b): OLS residual PACF
    plot_pacf(ols_resid, lags=DIAGNOSTIC_LAGS, alpha=.05, zero=False, method="ywm", ax=ax[0, 1])
    ax[0, 1].set_title("(b) OLS residual PACF (complete-case row lags)", fontweight="bold")
    ax[0, 1].set_xlabel("Row lag (complete cases)")
    ax[0, 1].set_ylabel("Partial Autocorrelation")

    # Panel (c): Final-model innovation ACF
    lags = np.arange(1, DIAGNOSTIC_LAGS + 1)
    ax[1, 0].stem(lags, r[1:], basefmt=" ")
    band = 1.96 / np.sqrt(max(1, int(np.nanmedian(pairs[1:]))))
    ax[1, 0].axhline(band, ls="--", lw=1, color="gray")
    ax[1, 0].axhline(-band, ls="--", lw=1, color="gray")
    ax[1, 0].axhline(0, color="black", lw=.7)
    ax[1, 0].set_ylim(-.5, 1)
    ax[1, 0].set_title("(c) Final-model innovation ACF (calendar preserved)", fontweight="bold")
    ax[1, 0].set_xlabel("Calendar lag (months)")
    ax[1, 0].set_ylabel("ACF")
    ax[1, 0].text(.03, .93, "Dashed band is approximate only\nNot a formal test",
                  transform=ax[1, 0].transAxes, va="top", fontsize=10, 
                  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

    # Panel (d): Final-model standardized innovations
    ax[1, 1].plot(cal.index, innov, lw=1.0, color="#1f77b4")
    ax[1, 1].axhline(0, color="black", lw=.7)
    ax[1, 1].set_title("(d) Final-model standardized innovations", fontweight="bold")
    ax[1, 1].set_xlabel("Month")
    ax[1, 1].set_ylabel("Standardized innovation")

    fig.suptitle("", fontsize=16, fontweight="bold", y=0.96)
    fig.tight_layout(rect=[0, 0.03, 1, 0.94])
    
    fig.savefig(OUT_PDF, bbox_inches="tight", dpi=300)
    fig.savefig(OUT_PNG, bbox_inches="tight", dpi=300)
    plt.close(fig)

    print(f"Figure successfully generated and saved to:\n- {OUT_PDF}\n- {OUT_PNG}")


if __name__ == "__main__":
    main()

Figure successfully generated and saved to:
- E:\Regional_Studies_Marine\Revision\Dataset_for_lag_test\MLRA_Outputs\GPT_code_output\MLR_residual_diagnostics_revised_2.pdf
- E:\Regional_Studies_Marine\Revision\Dataset_for_lag_test\MLRA_Outputs\GPT_code_output\MLR_residual_diagnostics_revised_2.png
